# 03 — Feature Engineering

> *"Attribute combinations... A useful feature might be created by combining existing attributes. For example, if you have the number of rooms and the number of households, it might be useful to calculate the number of rooms per household."*  
> — Aurélien Géron, *Hands-On Machine Learning with Scikit-Learn and PyTorch* (2025), Chapter 2

## 1. Why Feature Engineering?

Raw features rarely carry the signal in the form that models can best consume. Two guiding principles from Géron (Chapter 2) drive this step:

1. **Combine attributes** — ratios, products, and interaction terms often carry more discriminative power than either raw attribute alone.  
2. **Encode domain knowledge** — banking analysts know things the raw numbers cannot express by themselves. An age threshold for "senior" or the special risk pattern of customers with 3+ products are examples.

All engineered features in this project live in  
`src/client_churn_prediction/features.py` → `_churn_features()`.  

The public API is `prepare_features(df, config)`, which dispatches to the correct family function based on `config["project"]["family"]`.

## 2. Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

CHURN_PALETTE = {0: "#4C8CBF", 1: "#E84545"}

print("Libraries loaded.")

## 3. Load Raw Data

In [ ]:
from client_churn_prediction.config import load_config
from client_churn_prediction.data import load_training_frame

config = load_config(PROJECT_ROOT / "configs" / "project.toml")
df_raw = load_training_frame(config)

TARGET = "exited"

print(f"Raw dataset: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print("\nColumns before feature engineering:")
print(df_raw.columns.tolist())

## 4. Engineered Features — Design Rationale

Each feature below was motivated by observations from the EDA in notebook 02 and by domain expertise in retail banking. We document the *why* for each.

---

### 4.1 Balance-Based Signals

| Feature | Formula | Business rationale |
|---|---|---|
| `is_zero_balance` | `balance == 0` → 0/1 | ~30% of customers have zero balance; this is a structurally different segment and a non-linear effect the model should handle explicitly |
| `balance_per_product` | `balance / num_of_products` | A high balance spread across many products may signal over-committed customers vs. concentrated relationship |
| `balance_salary_ratio` | `balance / estimated_salary` | Relative wealth signal; a customer with balance = 10k and salary = 20k is more likely to churn than one with balance = 10k and salary = 200k |

### 4.2 Age / Tenure Signals

| Feature | Formula | Business rationale |
|---|---|---|
| `age_tenure_ratio` | `age / (tenure + 1)` | A 60-year-old with 1-year tenure joined late in life — higher churn risk than a 30-year-old with the same tenure |
| `is_senior` | `age >= 50` → 0/1 | EDA showed a clear age threshold around 50; discretisation preserves the step-change effect for linear models |

### 4.3 Engagement Signals

| Feature | Formula | Business rationale |
|---|---|---|
| `products_active_combo` | `num_of_products × is_active_member` | An inactive customer with many products is at higher risk than either signal alone suggests |
| `senior_inactive` | `(age >= 50) AND (is_active_member == 0)` → 0/1 | Interaction: older + inactive is the highest-risk sub-segment |

### 4.4 Credit Risk Segmentation

| Feature | Formula | Business rationale |
|---|---|---|
| `credit_score_band` | Ordinal bins: poor/fair/good/very_good/exceptional | Raw credit score has a near-uniform distribution; banding captures risk-category thresholds used by underwriters |

## 5. Apply Feature Engineering Pipeline

In [ ]:
from client_churn_prediction.features import prepare_features

df_eng = prepare_features(df_raw.copy(), config)

print(f"After feature engineering: {df_eng.shape[0]:,} rows x {df_eng.shape[1]} columns")
print(f"New columns added: {df_eng.shape[1] - df_raw.shape[1]}")

## 6. Before / After Comparison

In [ ]:
original_cols = set(df_raw.columns)
all_cols       = set(df_eng.columns)
new_cols       = sorted(all_cols - original_cols)

print(f"Original columns ({len(original_cols)}):")
print(sorted(original_cols))
print()
print(f"Engineered columns ({len(new_cols)}):")
print(new_cols)

In [ ]:
# Preview of new features alongside the target
preview_cols = new_cols + [TARGET]
df_eng[preview_cols].head(10)

In [ ]:
# Summary statistics of engineered numerical features
num_new = [c for c in new_cols if pd.api.types.is_numeric_dtype(df_eng[c])]
df_eng[num_new].describe().T.round(3)

## 7. Validate New Features — Churn Rate Analysis

A feature is useful if it creates a separation in the target variable. Let us verify that the engineered features actually separate churners from retained customers.

In [ ]:
# Binary engineered features — churn rate comparison
binary_eng = [c for c in new_cols if df_eng[c].nunique() <= 2 and pd.api.types.is_numeric_dtype(df_eng[c])]

rows = []
for col in binary_eng:
    group = df_eng.groupby(col)[TARGET].agg(["mean", "count"]).reset_index()
    for _, row in group.iterrows():
        rows.append({"feature": col, "value": row[col], "churn_rate": row["mean"] * 100, "n": row["count"]})

summary_df = pd.DataFrame(rows)
print("Churn rate by binary engineered features:")
print(summary_df.to_string(index=False))

In [ ]:
# Visual: churn rate for is_senior, senior_inactive, is_zero_balance side by side
binary_plot_cols = [c for c in ["is_senior", "senior_inactive", "is_zero_balance"] if c in df_eng.columns]

if binary_plot_cols:
    fig, axes = plt.subplots(1, len(binary_plot_cols), figsize=(5 * len(binary_plot_cols), 4))
    if len(binary_plot_cols) == 1:
        axes = [axes]

    for ax, col in zip(axes, binary_plot_cols):
        churn_by_flag = df_eng.groupby(col)[TARGET].mean() * 100
        ax.bar(
            ["No (0)", "Yes (1)"], churn_by_flag.values,
            color=[CHURN_PALETTE[0], CHURN_PALETTE[1]], edgecolor="white", width=0.45,
        )
        ax.axhline(df_eng[TARGET].mean() * 100, color="black", linestyle="--", linewidth=1, label="Overall avg")
        ax.set_title(col.replace("_", " ").title())
        ax.set_ylabel("Churn Rate (%)")
        ax.set_ylim(0, max(churn_by_flag.values) * 1.4)
        ax.legend(fontsize=8)
        for i, val in enumerate(churn_by_flag.values):
            ax.text(i, val + 0.5, f"{val:.1f}%", ha="center", fontsize=10, fontweight="bold")

    fig.suptitle("Churn Rate by Engineered Binary Features", fontsize=13, y=1.03)
    plt.tight_layout()
    plt.show()

In [ ]:
# credit_score_band churn rate
if "credit_score_band" in df_eng.columns:
    band_order = ["poor", "fair", "good", "very_good", "exceptional"]
    band_churn = (
        df_eng.groupby("credit_score_band")[TARGET]
        .agg(["mean", "count"])
        .reindex([b for b in band_order if b in df_eng["credit_score_band"].unique()])
    )

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(
        band_churn.index,
        band_churn["mean"] * 100,
        color=sns.color_palette("Blues", len(band_churn)),
        edgecolor="white",
    )
    ax.axhline(df_eng[TARGET].mean() * 100, color="red", linestyle="--", linewidth=1.2, label="Overall avg")
    ax.set_title("Churn Rate by Credit Score Band", fontsize=12)
    ax.set_xlabel("Credit Score Band")
    ax.set_ylabel("Churn Rate (%)")
    ax.legend()
    for bar, (_, row) in zip(bars, band_churn.iterrows()):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.3,
            f"{row['mean']*100:.1f}%\n(n={int(row['count']):,})",
            ha="center", fontsize=9,
        )
    plt.tight_layout()
    plt.show()

## 8. Feature Importance Preview (Quick Random Forest)

Following Géron's recommendation to run a quick baseline model during exploration — not for final evaluation, but to screen features and detect gross problems. We use a Random Forest with default settings on a train split.

Note: this is an *exploratory* importance chart, not the final model evaluation. Proper cross-validated evaluation happens in notebook 04.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# Drop identifier columns
id_cols = ["row_number", "customer_id", "surname", "row_id"]
drop_cols = [c for c in id_cols if c in df_eng.columns]

df_model = df_eng.drop(columns=drop_cols, errors="ignore").copy()

X = df_model.drop(columns=[TARGET])
y = df_model[TARGET]

# Identify column types for pipeline
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print(f"Numerical features : {len(num_cols)}")
print(f"Categorical features: {len(cat_cols)} -> {cat_cols}")
print(f"Total features      : {len(X.columns)}")

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Build a lightweight preprocessing + estimator pipeline
num_transformer = SimpleImputer(strategy="median")
cat_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

preprocessor = ColumnTransformer([
    ("num", num_transformer, num_cols),
    ("cat", cat_transformer, cat_cols),
], remainder="drop")

rf = Pipeline([
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )),
])

rf.fit(X_train, y_train)

from sklearn.metrics import roc_auc_score, average_precision_score

val_proba = rf.predict_proba(X_val)[:, 1]
print(f"Quick RF -- ROC AUC  : {roc_auc_score(y_val, val_proba):.4f}")
print(f"Quick RF -- Avg Prec : {average_precision_score(y_val, val_proba):.4f}")

In [ ]:
# Extract and plot feature importances
feature_names = num_cols + cat_cols
importances = rf.named_steps["clf"].feature_importances_

importance_df = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

# Mark engineered features
engineered_feature_set = set(new_cols)
importance_df["is_engineered"] = importance_df["feature"].isin(engineered_feature_set)

top_n = min(20, len(importance_df))
top_imp = importance_df.head(top_n)

fig, ax = plt.subplots(figsize=(9, 6))

bar_colors = ["#E84545" if eng else "#4C8CBF" for eng in top_imp["is_engineered"]]
ax.barh(
    top_imp["feature"][::-1],
    top_imp["importance"][::-1],
    color=bar_colors[::-1],
    edgecolor="white",
)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#E84545", label="Engineered feature"),
    Patch(facecolor="#4C8CBF", label="Original feature"),
]
ax.legend(handles=legend_elements, loc="lower right")

ax.set_title(f"Random Forest -- Feature Importances (Top {top_n})", fontsize=13)
ax.set_xlabel("Mean Decrease in Impurity")
ax.set_xlim(0, top_imp["importance"].max() * 1.15)
plt.tight_layout()
plt.show()

print("\nTop 10 features:")
print(importance_df.head(10).to_string(index=False))

## 9. Final Feature Set Summary

The following table consolidates the complete set of features that will be passed to the modelling pipeline in notebook 04.

In [ ]:
drop_ids = set(id_cols)
all_features = [c for c in df_eng.columns if c not in drop_ids and c != TARGET]

feature_summary = []
for col in all_features:
    dtype = str(df_eng[col].dtype)
    is_eng = col in engineered_feature_set
    n_unique = df_eng[col].nunique()
    missing_pct = df_eng[col].isna().mean() * 100
    if dtype not in ("object", "category"):
        corr_val = df_eng[[col, TARGET]].select_dtypes(include=np.number).corr().loc[col, TARGET]
        corr_str = round(float(corr_val), 3)
    else:
        corr_str = "N/A"
    feature_summary.append({
        "feature": col,
        "dtype": dtype,
        "engineered": is_eng,
        "n_unique": n_unique,
        "missing_%": round(missing_pct, 1),
        "corr_target": corr_str,
    })

feat_df = pd.DataFrame(feature_summary).sort_values(["engineered", "feature"], ascending=[False, True])
print(f"Total features for modelling: {len(feat_df)}")
print(f"  Original  : {(~feat_df['engineered']).sum()}")
print(f"  Engineered: {feat_df['engineered'].sum()}")
print()
feat_df

## 10. Key Takeaways

1. **`prepare_features(df, config)`** applies all domain-driven transformations in one call. The function is idempotent and dispatches by `config["project"]["family"]`, making this pipeline portable to other datasets in the template.

2. **Engineered features that appear in the top-10 importance list** confirm the EDA hypotheses:  
   - `age` / `is_senior` — the most consistently important individual signals.  
   - `balance_per_product`, `is_zero_balance` — balance signals contextualised by relationship depth.  
   - `senior_inactive` — the high-risk interaction between age and disengagement.

3. **Next step (notebook 04):** use this feature matrix in a proper cross-validated comparison of Logistic Regression, Random Forest, and Gradient Boosting — with hyperparameter search and final evaluation on Recall@k and Lift@k.

Following Géron's Chapter 2 advice: *"Do not think about what the best model might be at this point. Just explore and take notes."* The feature importance here is exploratory — final feature selection decisions will be validated in notebook 04 through cross-validation.